# p10 Lognormal · Spline-Ridge Mean-centered Huber-DML 全量实验

n=1000，p=10，Full R=1000，DGP1/DGP2，Gaussian 与 LN(0.25,0.5,0.75,1,1.25,1.5,2)。完整设计为 16,000 组数据、64,000 条方法记录。真实函数仅用前五列，十维 X 保持 AR(1) 相关结构。

**默认只运行 Smoke。正式运行由你将下面的 MODE 从 "smoke" 改成 "full"，再选择 environment2 内核运行全部单元。** Full 会先检查 Smoke，通过后按断点续跑。8 Worker × 1 数值线程。旧实验和断点全部保留。


## 固定方法与必要修改

M1/M2/M3/S 完整复用已验证的 p10 Student-t 实现。S 保留三次样条、5 个分位数节点、原线性项与全部交互项、原 RidgeCV 网格、OLS-MAD 阈值和算术均值位置 A 的 adjusted Score-test。只更换误差生成分支。

Lognormal 生成函数逐字复用 p5 正式实验，使用总体均值中心化、总体标准差标准化；sigma 是底层正态的标准差。保留原误差编号 8–14 和 seed 规则，不运行任何 Student-t 场景。

Smoke 检查十维输入、115 个特征、内外折隔离、真值隔离、串并行确定性和断点恢复；验证 Gaussian 历史数据/fold 哈希及四方法的点估计、检验统计量。历史复放仅用于校验，不写入正式断点。


In [ ]:
from pathlib import Path
import sys
import multiprocessing
from IPython.display import Image, Markdown, display

HERE = Path.cwd()
RESULTS = HERE / "Spline_Ridge_DML_p10_Lognormal_Full"
if not (RESULTS / "spline_p10_lognormal_worker.py").exists():
    HERE = Path.home() / "Desktop" / "毕设DML实验" / "DML_SplineRidge p10 Lognormal全量实验"
    RESULTS = HERE / "Spline_Ridge_DML_p10_Lognormal_Full"
sys.path.insert(0, str(RESULTS))
from spline_p10_lognormal_worker import config_for, smoke_run, full_run
MODE = "smoke"
RETRY_FAILED = False
assert MODE in ("smoke", "full")
settings = config_for(MODE)
print(f"p={settings['p']}，n={settings['n']}，R={settings['R']}，8 Worker × 1 数值线程")


## Smoke 自检

2 个 DGP × Gaussian、LN(0.25)、LN(1)、LN(2)，各 1 次，共 8 组、32 条方法记录。Coverage 不作为 Smoke 通过条件。

另外核对历史 p10 Gaussian 的 2,000 组数据/fold 哈希，固定复算每个 DGP 的第 0、499、999 次共 6 组四方法结果；这是隔离的自检，不启动 Full。


In [ ]:
if __name__ == "__main__":
    multiprocessing.freeze_support()
    smoke = smoke_run(RESULTS / "smoke")
    print(f"Smoke 通过，p={smoke['p']}，{smoke['data_n']} 组数据，"
          f"{smoke['point_n']} 个点估计和 {smoke['test_n']} 个检验有效。")


## 正式 Full R1000（仅 MODE="full" 时执行）

默认 Smoke 会跳过本节的正式计算。手动设为 full 后，按同一 Windows spawn/Worker 流程执行，使用独立 SQLite 断点，每 32 组提交事务并更新进度。只包含 Gaussian 和 Lognormal。

失败按方法记录，保留重试历史；RETRY_FAILED=True 可显式重试失败方法，不替换估计器。


In [ ]:
if __name__ == "__main__":
    if not smoke["passed"]:
        raise RuntimeError("Smoke 未通过")
    results = full_run(RESULTS, retry_failed=RETRY_FAILED) if MODE == "full" else None
    if results is not None:
        print(f"完成 {results['data_n']:,} 组数据；有效点估计 {results['point_n']:,}，检验 {results['test_n']:,}。")

    else:
        print("已完成 Smoke，Full 尚未启动。需要正式运行时将 MODE 改为 full，再运行全部单元。")


## 完整汇总、配对比较和图表

In [ ]:
if results is not None:
    display(results["summary"].round(6))
    display(results["paired"].round(6))
    for filename in ("coverage_lognormal.png", "deterioration_lognormal.png",
                     "bias_rmse_lognormal.png", "score_diagnostics.png",
                     "nuisance_diagnostics.png", "p10_vs_p5.png"):
        display(Image(filename=str(RESULTS / filename), width=1100))


## 中文报告、维度与误差分布比较

Full 完成后，输出全部标准 CSV、HTML、PNG。p10_vs_p5.png 展示 S 在两个 DGP 下的 Coverage、Bias、RMSE、Mean(T)、SD(T)。p10_tail_comparison.csv 比较本轮 LN(2) 与已完成的 p10 t(2.1)，不额外运行 Student-t。

不同维度和非 Gaussian 分布只进行场景级比较，不按 replication 配对。报告同时看竞争方法、点估计精度和 Gaussian 基准，不以 Coverage 是否恰好 95% 判断成功与失败。


In [ ]:
if results is not None:
    display(Markdown(results["findings"]))
